# ScpTensor 单细胞蛋白质组学分析入门教程

欢迎来到 ScpTensor 教程!ScpTensor 是一个专为单细胞蛋白质组学(Single-Cell Proteomics, SCP)数据分析设计的 Python 库,提供了完整的数据分析流程。

## 教程目标

通过本教程,你将学会:
- 加载和浏览单细胞蛋白质组学数据
- 进行质量控制和数据过滤
- 数据标准化和转换
- 缺失值填充
- 降维分析(PCA, UMAP)
- 聚类分析
- 批次效应校正
- 保存分析结果

## 数据集说明

本教程使用 T-SCP 数据集,这是一个细胞周期(Cell Cycle)相关的单细胞蛋白质组学数据集,包含不同细胞周期阶段(G1, G1-S, G2, G2-M)的细胞样本。

---

## 环境准备

首先,让我们导入必要的库并设置环境。

In [ ]:
# 核心库
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

# 设置绘图样式 (SciencePlots 风格,需要安装 SciencePlots 包)
try:
    plt.style.use(["science", "no-latex"])
    print("SciencePlots 样式加载成功!")
except Exception:
    # 如果 SciencePlots 未安装,使用默认样式
    print("提示: 安装 SciencePlots 包以获得更好的图表样式 (pip install SciencePlots)")

# ScpTensor 主要模块
# 忽略警告 (可选)
import warnings

from scptensor import (
    Assay,
    # 数据结构
    ScpContainer,
    ScpMatrix,
    calculate_feature_qc_metrics,
    # QC 函数
    calculate_sample_qc_metrics,
    # 聚类
    cluster_kmeans,
    filter_features_by_missingness,
    filter_low_quality_samples,
    # 缺失值填充
    impute_knn,
    # 标准化
    log_transform,
    norm_median,
    reduce_pca,
    reduce_umap,
    # I/O 函数
    save_hdf5,
)

warnings.filterwarnings("ignore")

print("ScpTensor 环境准备完成!")

---

# 第一章:数据加载

## 1.1 了解数据结构

ScpTensor 使用层级数据结构:
- **ScpContainer**: 顶层容器,存储样本元数据(obs)和多个 Assay
- **Assay**: 代表一个分析层(如蛋白质、肽段),存储特征元数据(var)和数据层(layers)
- **ScpMatrix**: 物理存储矩阵,包含数值(X)和掩码(M)

```
ScpContainer
|-- obs: 样本元数据 (n_samples x metadata)
|-- assays: Dict[str, Assay]
|   |-- Assay (如 'proteins')
|       |-- var: 特征元数据 (n_features x metadata)
|       |-- layers: Dict[str, ScpMatrix]
|           |-- ScpMatrix (如 'X', 'log', 'imputed')
|               |-- X: 数值矩阵
|               |-- M: 掩码矩阵
```

## 1.2 加载 CSV 数据

我们将使用 T-SCP 数据集,它包含两个文件:
- `T-SCP_data.csv`: 蛋白质表达矩阵 (样本 x 蛋白质)
- `T-SCP_cells.csv`: 样本元数据 (样本ID, 细胞周期阶段)

In [ ]:
# 数据路径
# 注意: 如果在 docs/ 目录运行,使用相对路径;否则调整路径
import os

if os.path.exists("../data/applications/cell_cycle/"):
    data_dir = "../data/applications/cell_cycle/"
else:
    data_dir = "data/applications/cell_cycle/"
data_file = f"{data_dir}T-SCP_data.csv"
meta_file = f"{data_dir}T-SCP_cells.csv"

# 使用 polars 加载数据
# 加载表达矩阵 (第一列是蛋白质ID,其余列是样本)
data_df = pl.read_csv(data_file)
print(f"表达矩阵维度: {data_df.shape}")
print("前5行预览:")
data_df.head()

In [ ]:
# 加载样本元数据
meta_df = pl.read_csv(meta_file)
print(f"样本元数据维度: {meta_df.shape}")
print("细胞周期阶段分布:")
print(meta_df["cell_cycle"].value_counts())
meta_df.head()

## 1.3 创建 ScpContainer

现在我们将数据组织成 ScpContainer 结构。

In [ ]:
# 提取样本ID和蛋白质ID
# 数据格式: 第一列是样本ID, 第一行是蛋白质ID
# 第一列名称为空 (index column)
protein_ids = data_df.columns[1:]  # 跳过第一列(样本ID列), columns 返回 list
sample_ids = data_df[data_df.columns[0]].to_list()  # 第一列是样本ID

# 提取表达矩阵 (跳过第一列样本ID)
X = data_df.select(data_df.columns[1:]).to_numpy().astype(np.float64)

# 将0值替换为NaN (在单细胞蛋白质组学中,0通常表示缺失)
X[X == 0] = np.nan

print(f"表达矩阵形状: {X.shape} (样本数 x 蛋白质数)")
print(f"缺失值数量: {np.isnan(X).sum()}")
print(f"缺失值比例: {np.isnan(X).sum() / X.size * 100:.2f}%")

In [ ]:
# 创建样本元数据 DataFrame
# 从 meta_df 获取细胞周期信息,与样本ID对齐
obs = pl.DataFrame({"_index": sample_ids})

# 从 meta_df 添加细胞周期信息
meta_index = meta_df[meta_df.columns[0]].to_list()
meta_cycle = meta_df["cell_cycle"].to_list()
cycle_dict = dict(zip(meta_index, meta_cycle, strict=False))
obs = obs.with_columns(
    pl.col("_index").replace_strict(cycle_dict, default="Unknown").alias("cell_cycle")
)

# 创建特征(蛋白质)元数据 DataFrame
var = pl.DataFrame({"_index": protein_ids})

# 创建 ScpMatrix
matrix = ScpMatrix(X=X)

# 创建 Assay
assay = Assay(var=var, layers={"raw": matrix}, feature_id_col="_index")

# 创建 ScpContainer
container = ScpContainer(obs=obs, assays={"proteins": assay}, sample_id_col="_index")

print("ScpContainer 创建成功!")
print(container)

In [ ]:
# 查看容器基本信息
print(f"样本数量: {container.n_samples}")
print(f"Assay 数量: {len(container.assays)}")
print(f"Assay 名称: {list(container.assays.keys())}")

# 查看 Assay 信息
assay = container.assays["proteins"]
print(f"\n蛋白质数量: {assay.n_features}")
print(f"数据层: {list(assay.layers.keys())}")

In [ ]:
# 查看样本元数据
print("样本元数据 (obs):")
print(container.obs.head())

In [ ]:
# 查看特征元数据
print("特征元数据 (var):")
print(container.assays["proteins"].var.head())

---

# 第二章:质量控制 (QC)

质量控制是单细胞分析的关键步骤。ScpTensor 提供了多层级的 QC 功能:
- **样本级别**: 检测低质量样本、双细胞
- **特征级别**: 过滤高缺失率、高变异系数的蛋白质

## 2.1 样本质控

使用 `calculate_sample_qc_metrics()` 计算每个样本的质量指标。

In [ ]:
# 计算样本 QC 指标
container = calculate_sample_qc_metrics(container, assay_name="proteins", layer_name="raw")

print("样本 QC 指标计算完成!")
print(f"obs 列: {container.obs.columns}")

In [ ]:
# 查看 QC 结果
qc_cols = [c for c in container.obs.columns if "proteins" in c]
print("QC 相关列:")
print(container.obs.select(["_index", "cell_cycle"] + qc_cols).head(10))

In [ ]:
# 可视化样本 QC 指标
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 检出蛋白数量分布
axes[0].hist(container.obs["n_features_proteins"].to_numpy(), bins=30, edgecolor="black")
axes[0].set_xlabel("Detected Proteins")
axes[0].set_ylabel("Number of Samples")
axes[0].set_title("Distribution of Detected Proteins")

# 缺失率分布
# 计算缺失率
n_total_features = container.assays["proteins"].n_features
missing_ratio = 1 - container.obs["n_features_proteins"].to_numpy() / n_total_features
axes[1].hist(missing_ratio, bins=30, edgecolor="black", color="orange")
axes[1].set_xlabel("Missing Value Ratio")
axes[1].set_ylabel("Number of Samples")
axes[1].set_title("Distribution of Missing Value Ratio")

# 总强度分布
axes[2].hist(
    np.log10(container.obs["total_intensity_proteins"].to_numpy() + 1),
    bins=30,
    edgecolor="black",
    color="green",
)
axes[2].set_xlabel("Log10(Total Intensity + 1)")
axes[2].set_ylabel("Number of Samples")
axes[2].set_title("Distribution of Total Intensity")

plt.tight_layout()
plt.savefig("sample_qc_metrics.png", dpi=300)
plt.show()
print("图表已保存至 sample_qc_metrics.png")

## 2.2 特征质控

使用 `calculate_feature_qc_metrics()` 计算每个蛋白质的质量指标。

In [ ]:
# 计算特征 QC 指标
container = calculate_feature_qc_metrics(container, assay_name="proteins", layer_name="raw")

print("特征 QC 指标计算完成!")
assay = container.assays["proteins"]
print(f"var 列: {assay.var.columns}")

In [ ]:
# 查看特征 QC 结果
qc_cols = ["missing_rate", "detection_rate", "mean_expression", "cv"]
print("特征 QC 列:")
print(assay.var.select(["_index"] + qc_cols).head(10))

In [ ]:
# 可视化特征 QC 指标
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 特征缺失率分布
axes[0].hist(assay.var["missing_rate"].to_numpy(), bins=50, edgecolor="black")
axes[0].set_xlabel("Missing Value Ratio")
axes[0].set_ylabel("Number of Proteins")
axes[0].set_title("Distribution of Protein Missingness")
axes[0].axvline(x=0.5, color="red", linestyle="--", label="50% threshold")
axes[0].legend()

# 特征变异系数分布
cv_values = assay.var["cv"].to_numpy()
cv_values = cv_values[~np.isnan(cv_values)]  # 移除 NaN
axes[1].hist(cv_values, bins=50, edgecolor="black", color="purple")
axes[1].set_xlabel("Coefficient of Variation (CV)")
axes[1].set_ylabel("Number of Proteins")
axes[1].set_title("Distribution of Protein CV")

plt.tight_layout()
plt.savefig("feature_qc_metrics.png", dpi=300)
plt.show()
print("图表已保存至 feature_qc_metrics.png")

---

# 第三章:数据过滤

根据 QC 结果过滤低质量的样本和特征。

## 3.1 过滤低质量样本

使用 `filter_low_quality_samples()` 过滤质量较差的样本。

In [ ]:
# 过滤前的样本数量
n_samples_before = container.n_samples
print(f"过滤前样本数量: {n_samples_before}")

# 过滤低质量样本
# 参数说明:
# - min_detected: 最少检出的特征数
# - max_missing_ratio: 最大允许缺失率
container = filter_low_quality_samples(
    container,
    assay_name="proteins",
    min_features=100,  # 至少检出 100 个蛋白
    use_mad=True,  # 使用 MAD 离群值检测
    nmads=3.0,  # MAD 阈值
)

n_samples_after = container.n_samples
print(f"过滤后样本数量: {n_samples_after}")
print(f"移除样本数: {n_samples_before - n_samples_after}")

## 3.2 过滤高缺失特征

使用 `filter_features_by_missingness()` 过滤缺失率过高的蛋白质。

In [ ]:
# 过滤前的特征数量
n_features_before = container.assays["proteins"].n_features
print(f"过滤前蛋白质数量: {n_features_before}")

# 过滤高缺失特征
container = filter_features_by_missingness(
    container,
    assay_name="proteins",
    layer_name="raw",
    max_missing_rate=0.5,  # 缺失率不超过 50%
)

n_features_after = container.assays["proteins"].n_features
print(f"过滤后蛋白质数量: {n_features_after}")
print(f"移除蛋白质数: {n_features_before - n_features_after}")

In [ ]:
# 过滤后的数据概况
X = container.assays["proteins"].layers["raw"].X
print("\n过滤后数据概况:")
print(f"矩阵形状: {X.shape}")
print(f"缺失值比例: {np.isnan(X).sum() / X.size * 100:.2f}%")

---

# 第四章: 数据预处理与标准化

**重要概念：正确的处理流程**

在蛋白质组学数据分析中，数据预处理应遵循以下顺序：

```
原始数据 (raw) → 对数转换 (log_transform) → 标准化 (normalization)
```

**1. 对数转换（预处理步骤）**
- 应该**首先**应用于原始强度值
- 作用：稳定方差、使数据更接近正态分布
- 函数：`log_transform()`

**2. 标准化（校正步骤）**
- 应用于**对数转换后**的数据
- 作用：消除样本间的技术变异
- 可选方法：
  - `norm_median()`: 中位数标准化（稳健，适合有离群值的数据）
  - `norm_mean()`: 均值标准化
  - `norm_quantile()`: 分位数标准化（使分布完全匹配）

## 4.1 对数转换（预处理）

对数转换是数据预处理的**第一步**，应该直接应用于原始强度值。

对数转换的作用：
1. 稳定方差（使高值和低值的变异更均匀）
2. 使数据分布更接近正态分布
3. 减少极端值的影响
4. 将强度值转换到更适合统计分析的尺度

In [ ]:
# 步骤 1: 应用对数转换（预处理）
# 注意：对数转换应该首先应用于原始数据
container = log_transform(
    container,
    assay_name="proteins",
    source_layer="raw",
    new_layer_name="log",
    base=2.0,
    offset=1.0,  # log2(x + 1)
)

print("对数转换完成！")
print(f"可用层: {list(container.assays['proteins'].layers.keys())}")

## 4.2 中位数标准化

在对数转换**之后**，应用标准化方法来消除样本间的技术变异。

中位数标准化的特点：
- 消除样本间的全局强度差异
- 对离群值稳健（比均值标准化更稳健）
- 适合大多数蛋白质组学数据

In [ ]:
# 步骤 2: 应用中位数标准化
# 注意：标准化应用于对数转换后的数据
container = norm_median(
    container,
    assay_name="proteins",
    source_layer="log",
    new_layer_name="normalized",
)

print("中位数标准化完成！")
print(f"可用层: {list(container.assays['proteins'].layers.keys())}")

# 其他标准化方法示例（根据需要选择一种）：
# container = norm_mean(
#     container,
#     assay_name="proteins",
#     source_layer="log",
#     new_layer_name="normalized",
# )
#
# container = norm_quantile(
#     container,
#     assay_name="proteins",
#     source_layer="log",
#     new_layer_name="normalized",
# )

In [ ]:
# 比较预处理和标准化前后的数据分布
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 原始数据
X_raw = container.assays["proteins"].layers["raw"].X.flatten()
X_raw = X_raw[~np.isnan(X_raw)]
axes[0].hist(np.log10(X_raw + 1), bins=50, edgecolor="black")
axes[0].set_xlabel("Log10(Intensity + 1)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Raw Data Distribution")

# 对数转换后
X_log = container.assays["proteins"].layers["log"].X.flatten()
X_log = X_log[~np.isnan(X_log)]
axes[1].hist(X_log, bins=50, edgecolor="black", color="orange")
axes[1].set_xlabel("Log2(Intensity + 1)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("After Log Transform (Preprocessing)")

# 标准化后
X_norm = container.assays["proteins"].layers["normalized"].X.flatten()
X_norm = X_norm[~np.isnan(X_norm)]
axes[2].hist(X_norm, bins=50, edgecolor="black", color="green")
axes[2].set_xlabel("Log2 Normalized Intensity")
axes[2].set_ylabel("Frequency")
axes[2].set_title("After Median Normalization")

plt.tight_layout()
plt.savefig("normalization_comparison.png", dpi=300)
plt.show()
print("图表已保存至 normalization_comparison.png")

---

# 第五章:缺失值填充

单细胞蛋白质组学数据通常有大量缺失值。ScpTensor 提供了多种填充方法:

| 方法 | 函数 | 特点 | 适用场景 |
|------|------|------|----------|
| KNN | `impute_knn()` | 基于最近邻 | 通用 |
| BPCA | `impute_bpca()` | 贝叶斯 PCA | 低缺失率数据 |
| MissForest | `impute_mf()` | 随机森林 | 高维数据 |
| LLS | `impute_lls()` | 局部最小二乘 | 有相关性数据 |
| QRILC | `impute_qrilc()` | 分位数回归 | 左截断数据 |
| MinProb | `impute_minprob()` | 概率最小值 | MNAR 缺失 |

## 5.1 KNN 填充

K-近邻(KNN)填充使用相似样本的值来估计缺失值。

In [ ]:
# 应用 KNN 填充
container = impute_knn(
    container,
    assay_name="proteins",
    source_layer="log",
    new_layer_name="imputed",
    k=5,  # 使用 5 个最近邻
    weights="distance",  # 距离加权
)

print("KNN 填充完成!")
print(f"可用层: {list(container.assays['proteins'].layers.keys())}")

In [ ]:
# 检查填充结果
X_imputed = container.assays["proteins"].layers["imputed"].X
print(f"填充后矩阵形状: {X_imputed.shape}")
print(f"剩余缺失值: {np.isnan(X_imputed).sum()}")

## 5.2 其他填充方法介绍

除了 KNN,还可以尝试其他方法:

In [ ]:
# BPCA 填充示例 (适合低缺失率数据)
# container = impute_bpca(
#     container,
#     assay_name='proteins',
#     source_layer='log',
#     new_layer_name='bpca_imputed',
#     n_components=10
# )

# MissForest 填充示例 (适合高维数据)
# container = impute_mf(
#     container,
#     assay_name='proteins',
#     source_layer='log',
#     new_layer_name='mf_imputed',
#     n_estimators=100,
#     max_iterations=10
# )

print("其他填充方法已注释,可根据需要取消注释运行。")

In [ ]:
# 可视化填充效果
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 填充前 (log 数据)
X_log = container.assays["proteins"].layers["log"].X
missing_mask = np.isnan(X_log)
axes[0].imshow(missing_mask[:50, :50], aspect="auto", cmap="viridis")
axes[0].set_xlabel("Proteins")
axes[0].set_ylabel("Samples")
axes[0].set_title("Missing Values Before Imputation\n(Yellow=Missing)")

# 填充后
X_imputed = container.assays["proteins"].layers["imputed"].X
axes[1].imshow(X_imputed[:50, :50], aspect="auto", cmap="RdBu_r")
axes[1].set_xlabel("Proteins")
axes[1].set_ylabel("Samples")
axes[1].set_title("Data After KNN Imputation")

plt.tight_layout()
plt.savefig("imputation_comparison.png", dpi=300)
plt.show()
print("图表已保存至 imputation_comparison.png")

> **提示**: 填充方法的选择取决于数据特点和缺失机制。建议尝试多种方法并比较结果。

---

# 第六章:降维分析

降维有助于数据可视化和后续分析。ScpTensor 支持:
- **PCA**: 线性降维,保留最大方差
- **UMAP**: 非线性降维,保留局部结构

## 6.1 PCA 降维

In [ ]:
# 运行 PCA
container = reduce_pca(
    container,
    assay_name="proteins",
    base_layer="imputed",
    n_components=50,  # 计算 50 个主成分
    new_assay_name="pca",
)

print("PCA 降维完成!")
print(f"可用 assays: {list(container.assays.keys())}")

In [ ]:
# 查看方差解释率
pca_assay = container.assays["pca"]
pca_var = pca_assay.var

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 单个主成分方差解释
var_ratio = pca_var["explained_variance_ratio"].to_numpy()
axes[0].bar(range(1, 21), var_ratio[:20], edgecolor="black")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Variance Explained")
axes[0].set_title("Variance Explained by Each PC")

# 累积方差解释
cumsum = np.cumsum(var_ratio)
axes[1].plot(range(1, 51), cumsum[:50], "b-o", markersize=4)
axes[1].axhline(y=0.8, color="red", linestyle="--", label="80% threshold")
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Variance Explained")
axes[1].set_title("Cumulative Variance Explained")
axes[1].legend()

plt.tight_layout()
plt.savefig("pca_variance.png", dpi=300)
plt.show()
print("图表已保存至 pca_variance.png")

In [ ]:
# 可视化 PCA 结果 (按细胞周期着色)
X_pca = container.assays["pca"].layers["X"].X
cell_cycle = container.obs["cell_cycle"].to_numpy()

# 创建颜色映射
unique_phases = np.unique(cell_cycle)
colors = {"G1": "blue", "G1-S": "green", "G2": "orange", "G2-M": "red"}

fig, ax = plt.subplots(figsize=(8, 6))
for phase in unique_phases:
    mask = cell_cycle == phase
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1], c=colors.get(phase, "gray"), label=phase, alpha=0.6, s=30
    )

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA of Single-Cell Proteomics Data")
ax.legend()
plt.tight_layout()
plt.savefig("pca_plot.png", dpi=300)
plt.show()
print("图表已保存至 pca_plot.png")

## 6.2 UMAP 降维

In [ ]:
# 运行 UMAP
container = reduce_umap(
    container,
    assay_name="pca",  # 基于 PCA assay
    base_layer="X",
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    new_assay_name="umap",
)

print("UMAP 降维完成!")
print(f"可用 assays: {list(container.assays.keys())}")

In [ ]:
# 可视化 UMAP 结果
X_umap = container.assays["umap"].layers["X"].X
cell_cycle = container.obs["cell_cycle"].to_numpy()

fig, ax = plt.subplots(figsize=(8, 6))
for phase in unique_phases:
    mask = cell_cycle == phase
    ax.scatter(
        X_umap[mask, 0], X_umap[mask, 1], c=colors.get(phase, "gray"), label=phase, alpha=0.6, s=30
    )

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_title("UMAP of Single-Cell Proteomics Data")
ax.legend()
plt.tight_layout()
plt.savefig("umap_plot.png", dpi=300)
plt.show()
print("图表已保存至 umap_plot.png")

---

# 第七章:聚类分析

聚类可以发现数据中的细胞亚群。ScpTensor 提供:
- `cluster_kmeans()`: K-means 聚类
- `cluster_leiden()`: Leiden 图聚类 (需要邻接图)

## 7.1 K-means 聚类

In [ ]:
# 运行 K-means 聚类
# 假设我们预期 4 个细胞周期阶段
container = cluster_kmeans(
    container, assay_name="pca", base_layer="X", n_clusters=4, key_added="kmeans_cluster"
)

print("K-means 聚类完成!")
print("聚类结果已保存到 obs['kmeans_cluster']")

In [ ]:
# 查看聚类结果
print("聚类分布:")
print(container.obs["kmeans_cluster"].value_counts())

In [ ]:
# 可视化聚类结果 (UMAP)
X_umap = container.assays["umap"].layers["X"].X
# Convert cluster labels to integers for plotting
clusters = container.obs["kmeans_cluster"].to_numpy().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 按聚类着色
scatter1 = axes[0].scatter(X_umap[:, 0], X_umap[:, 1], c=clusters, cmap="tab10", alpha=0.6, s=30)
axes[0].set_xlabel("UMAP1")
axes[0].set_ylabel("UMAP2")
axes[0].set_title("K-means Clustering (k=4)")
plt.colorbar(scatter1, ax=axes[0], label="Cluster")

# 按细胞周期着色 (真实标签)
cell_cycle = container.obs["cell_cycle"].to_numpy()
for phase in unique_phases:
    mask = cell_cycle == phase
    axes[1].scatter(
        X_umap[mask, 0], X_umap[mask, 1], c=colors.get(phase, "gray"), label=phase, alpha=0.6, s=30
    )
axes[1].set_xlabel("UMAP1")
axes[1].set_ylabel("UMAP2")
axes[1].set_title("True Cell Cycle Phases")
axes[1].legend()

plt.tight_layout()
plt.savefig("clustering_comparison.png", dpi=300)
plt.show()
print("图表已保存至 clustering_comparison.png")

## 7.2 Leiden 聚类 (可选)

Leiden 聚类是一种基于图的聚类方法,能自动确定聚类数量。

In [ ]:
# Leiden 聚类示例
# container = cluster_leiden(
#     container,
#     assay_name='pca',
#     base_layer='X',
#     n_neighbors=15,
#     resolution=0.5,
#     key_added='leiden_cluster'
# )
#
# print("Leiden 聚类完成!")
# print(container.obs['leiden_cluster'].value_counts())

print("Leiden 聚类代码已注释,可根据需要取消注释运行。")

---

# 第八章:批次效应校正 (可选)

当数据来自多个批次时,批次效应校正可以消除技术变异。ScpTensor 支持:
- `integrate_combat()`: ComBat (经验贝叶斯)
- `integrate_harmony()`: Harmony
- `integrate_mnn()`: 互最近邻
- `integrate_scanorama()`: Scanorama

In [ ]:
# ComBat 批次校正示例
# 需要在 obs 中有批次信息列

# 添加模拟批次信息 (示例)
# container = container.with_columns(
#     batch=pl.when(pl.col("_index").str.starts_with("NB"))
#     .then(pl.lit("batch1"))
#     .otherwise(pl.lit("batch2"))
# )

# container = integrate_combat(
#     container,
#     assay_name='proteins',
#     source_layer='log',
#     batch_key='batch',
#     new_layer_name='combat'
# )

print("批次校正代码已注释,可根据需要取消注释运行。")

---

# 第九章:保存结果

分析完成后,保存结果以便后续使用。ScpTensor 支持多种格式:
- **HDF5**: 推荐格式,完整保存所有数据
- **CSV**: 人类可读,便于分享
- **NPZ**: 快速读写

## 9.1 保存为 HDF5 格式

In [ ]:
# 保存为 HDF5
output_hdf5 = "scp_analysis_results.h5"

save_hdf5(container, output_hdf5, compression="gzip", save_history=True, overwrite=True)

print(f"结果已保存至: {output_hdf5}")

## 9.2 保存为 CSV 格式

In [ ]:
# 保存为 CSV 目录
import os

import polars as pl

output_csv_dir = "scp_analysis_results/"

# 保存 proteins assay 的 imputed 层
os.makedirs(output_csv_dir, exist_ok=True)

# 保存样本元数据
container.obs.write_csv(f"{output_csv_dir}/obs.csv")

# 保存 proteins assay 的 imputed 数据
assay = container.assays["proteins"]
X_imputed = assay.layers["imputed"].X
sample_ids = container.sample_ids.to_list()
feature_ids = assay.feature_ids.to_list()

data_dict = {"_index": sample_ids}
for i, fid in enumerate(feature_ids):
    data_dict[str(fid)] = X_imputed[:, i]

data_df = pl.DataFrame(data_dict)
data_df.write_csv(f"{output_csv_dir}/proteins_imputed.csv")

print(f"结果已保存至: {output_csv_dir}")

In [ ]:
# 查看保存的文件
import os

print("CSV 输出文件:")
for f in os.listdir(output_csv_dir):
    print(f"  - {f}")

## 9.3 加载保存的数据

In [ ]:
# 从 CSV 加载
# 这里演示如何手动加载保存的数据
import polars as pl

loaded_obs = pl.read_csv(f"{output_csv_dir}/obs.csv")
loaded_data = pl.read_csv(f"{output_csv_dir}/proteins_imputed.csv")

print(f"加载的样本数: {len(loaded_obs)}")
print(f"加载的特征数: {len(loaded_data.columns) - 1}")  # 减去 _index 列

# 总结

恭喜你完成了 ScpTensor 教程！让我们回顾一下完整的分析流程：

## 完整分析流程回顾

```python
# 1. 数据加载
container = create_container_from_csv(...)

# 2. 质量控制
container = calculate_sample_qc_metrics(container, ...)
container = calculate_feature_qc_metrics(container, ...)

# 3. 数据过滤
container = filter_low_quality_samples(container, ...)
container = filter_features_by_missingness(container, ...)

# 4. 数据预处理与标准化（正确的顺序！）
container = log_transform(container, source_layer='raw', new_layer_name='log')
container = norm_median(container, source_layer='log', new_layer_name='normalized')
# 或者选择其他标准化方法：norm_mean(), norm_quantile()

# 5. 缺失值填充
container = impute_knn(container, source_layer='normalized', ...)

# 6. 降维
container = reduce_pca(container, ...)
container = reduce_umap(container, ...)

# 7. 聚类
container = cluster_kmeans(container, ...)

# 8. 保存结果
save_hdf5(container, "results.h5")
```

## 常用函数速查表

| 功能 | 函数 | 说明 |
|------|------|------|
| **数据加载** | `load_csv()` | 加载 CSV 格式数据 |
| | `load_hdf5()` | 加载 HDF5 格式数据 |
| **质量控制** | `calculate_sample_qc_metrics()` | 计算样本 QC 指标 |
| | `calculate_feature_qc_metrics()` | 计算特征 QC 指标 |
| **过滤** | `filter_low_quality_samples()` | 过滤低质量样本 |
| | `filter_features_by_missingness()` | 过滤高缺失特征 |
| | `filter_features_by_cv()` | 过滤高 CV 特征 |
| **预处理** | `log_transform()` | **第一步**：对数转换（预处理） |
| **标准化** | `norm_median()` | **第二步**：中位数标准化 |
| | `norm_mean()` | 第二步：均值标准化 |
| | `norm_quantile()` | 第二步：分位数标准化 |
| **填充** | `impute_knn()` | KNN 填充 |
| | `impute_bpca()` | BPCA 填充 |
| | `impute_mf()` | MissForest 填充 |
| **降维** | `reduce_pca()` | PCA 降维 |
| | `reduce_umap()` | UMAP 降维 |
| **聚类** | `cluster_kmeans()` | K-means 聚类 |
| | `cluster_leiden()` | Leiden 聚类 |
| **批次校正** | `integrate_combat()` | ComBat 校正 |
| | `integrate_harmony()` | Harmony 校正 |
| **保存** | `save_hdf5()` | 保存 HDF5 |
| | `save_csv()` | 保存 CSV |

**重要**: 处理顺序必须是 `raw` → `log_transform` → `normalization` → `imputation`

## 常见问题与技巧

### 1. 数据加载问题
- 确保 CSV 文件格式正确，第一行为列名
- 检查缺失值的表示方式 (NaN, NA, 空字符串)

### 2. 质量控制建议
- 先可视化 QC 指标分布，再选择阈值
- 不要过度过滤，保留足够的数据

### 3. 数据预处理与标准化的正确顺序
- **第一步：对数转换** - 应用于原始数据，稳定方差
- **第二步：标准化** - 应用于对数转换后的数据，消除技术变异
- **重要**：不要颠倒顺序！对数转换必须是第一步

### 4. 标准化方法选择
- **中位数标准化**：稳健，适合有离群值的数据（推荐）
- **均值标准化**：适合分布均匀的数据
- **分位数标准化**：适合需要完全匹配分布的情况

### 5. 填充方法选择
- KNN: 通用，适合大多数情况
- BPCA: 适合低缺失率数据
- MinProb: 适合 MNAR (非随机缺失) 情况

### 6. 聚类参数调优
- K-means: 需要预先指定 k 值，可使用肘部法则确定
- Leiden: 自动确定聚类数，通过 resolution 参数控制粒度

---

## 下一步

完成本教程后,你可以:
1. 探索更多可视化函数 (`scptensor.viz`)
2. 尝试差异表达分析 (`scptensor.diff_expr`)
3. 使用基准测试工具评估不同方法 (`scptensor.benchmark`)
4. 阅读官方文档了解更多高级功能

感谢使用 ScpTensor!如有问题,请在 GitHub 提交 Issue。